In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, confusion_matrix, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path 

In [3]:
# ── Load gap-free price panel (no coverage gaps) ──────────────────────
prices = pd.read_parquet(
    REPO_ROOT / "data" / "processed" / "prices_daily_accumulated.parquet"
)
prices["date"] = pd.to_datetime(prices["date"])
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
prices["ret_close"] = prices.groupby("ticker")["close"].pct_change()

g = prices.groupby("ticker")

# Momentum (all lagged by 1+ days — no leakage)
prices["mom_1d"]     = g["ret_close"].shift(1)
prices["mom_2d"]     = g["ret_close"].shift(2)
prices["mom_5d_avg"] = g["ret_close"].transform(lambda x: x.shift(1).rolling(5).mean())

# Volatility
prices["vol_10d"] = g["ret_close"].transform(lambda x: x.shift(1).rolling(10).std())
prices["vol_20d"] = g["ret_close"].transform(lambda x: x.shift(1).rolling(20).std())

# Volume z-score (using yesterday's volume, not today's)
prices["volume_z"] = (
    (g["volume"].shift(1) - g["volume"].transform(lambda x: x.shift(1).rolling(10).mean()))
    / g["volume"].transform(lambda x: x.shift(1).rolling(10).std())
)

# RSI-14 — shifted by 1 so it uses yesterday's close as latest
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

prices["rsi_14"] = (
    g["close"].transform(lambda x: compute_rsi(x, 14))
    .groupby(prices["ticker"]).shift(1)
)

# Distance from 10-day MA (using yesterday's close, yesterday's MA)
prices["ma_10"] = g["close"].transform(lambda x: x.rolling(10).mean()).groupby(prices["ticker"]).shift(1)
prices["close_prev"] = g["close"].shift(1)
prices["dist_from_ma10"] = (prices["close_prev"] - prices["ma_10"]) / prices["ma_10"]

print(f"Price panel: {len(prices):,} rows, {prices['ticker'].nunique()} tickers")
print(f"Date range: {prices['date'].min()} -> {prices['date'].max()}")

Price panel: 3,584 rows, 7 tickers
Date range: 2024-02-08 00:00:00 -> 2026-02-24 00:00:00


In [4]:
# ── Merge gap-free price features into ticker_daily ───────────────────
price_features = prices[[
    "date", "ticker",
    "mom_1d", "mom_2d", "mom_5d_avg",
    "vol_10d", "vol_20d", "volume_z",
    "rsi_14", "dist_from_ma10",
]].rename(columns={"date": "price_date"})

td = ticker_daily.merge(price_features, on=["price_date", "ticker"], how="left")

# ── Sentiment dynamics ────────────────────────────────────────────────
td["sent_5d_avg"]   = td.groupby("ticker")["mean_sent"].transform(lambda x: x.shift(1).rolling(5).mean())
td["sent_surprise"] = td["mean_sent"] - td["sent_5d_avg"]
td["sent_mom_3d"]   = td.groupby("ticker")["mean_sent"].transform(lambda x: x.diff(3))

# ── Extreme sentiment flags ──────────────────────────────────────────
td["neg_extreme_20"] = (td["mean_sent"] <= td["mean_sent"].quantile(0.20)).astype(float)

# ── Interaction terms (sentiment x momentum) ─────────────────────────
td["neg20_x_mom1d"]      = td["neg_extreme_20"] * td["mom_1d"]
td["sent_x_mom1d"]       = td["mean_sent"] * td["mom_1d"]
td["neg_ratio_x_mom1d"]  = td.get("neg_ratio", td["neg_extreme_20"]) * td["mom_1d"]

# ── Composite score: weighted sum of 7 significant signals ───────────
sig_features = ["rsi_14", "mom_1d", "volume_z", "sent_mom_3d",
                "neg_ratio_x_mom1d", "dist_from_ma10", "sent_x_mom1d"]
signal_weights = {
    "rsi_14": -0.062, "mom_1d": -0.055, "volume_z": +0.054,
    "sent_mom_3d": -0.054, "neg_ratio_x_mom1d": -0.051,
    "dist_from_ma10": -0.049, "sent_x_mom1d": +0.047,
}

for f in sig_features:
    td[f"z_{f}"] = (td[f] - td[f].mean()) / td[f].std()

td["composite"] = sum(signal_weights[f] * td[f"z_{f}"] for f in sig_features)

# ── Market-relative return (alternative target) ──────────────────────
mkt_ret = td.groupby("price_date")["ret"].mean().rename("mkt_ret").reset_index()
td = td.merge(mkt_ret, on="price_date", how="left")
td["relative_ret"] = td["ret"] - td["mkt_ret"]
td["beat_market"]  = (td["relative_ret"] > 0).astype(int)

# ── Ticker encoding ──────────────────────────────────────────────────
td["ticker_code"] = td["ticker"].astype("category").cat.codes

# ── Targets ───────────────────────────────────────────────────────────
td["up"] = (td["ret"] > 0).astype(int)

print(f"Final table: {len(td):,} rows")
print(f"Composite signal: rho={spearmanr(td['composite'].dropna(), td.loc[td['composite'].notna(), 'ret'])[0]:+.4f}")

NameError: name 'ticker_daily' is not defined

In [6]:
# 1. mom_1d: yesterday's return
ticker_daily['mom_1d'] = (
    ticker_daily.groupby('ticker')['ret']
    .transform(lambda x: x.shift(1))
)

# 2. mom_5d: 5-day average momentum
ticker_daily['mom_5d'] = (
    ticker_daily.groupby('ticker')['ret']
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

# 3. vol_10d: 10-day rolling volatility
ticker_daily['vol_10d'] = (
    ticker_daily.groupby('ticker')['ret']
    .transform(lambda x: x.shift(1).rolling(10).std())
)

# 4. mean_sent: already computed in aggregation

# 5. sent_5d_avg: 5-day rolling sentiment average
ticker_daily['sent_5d_avg'] = (
    ticker_daily.groupby('ticker')['mean_sent']
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

# 6. sent_surprise: today's sentiment vs recent average
ticker_daily['sent_surprise'] = ticker_daily['mean_sent'] - ticker_daily['sent_5d_avg']

# 7. volume_z: standardized volume
ticker_daily['vol_10d_mean'] = (
    ticker_daily.groupby('ticker')['volume']
    .transform(lambda x: x.shift(1).rolling(10).mean())
)
ticker_daily['vol_10d_std'] = (
    ticker_daily.groupby('ticker')['volume']
    .transform(lambda x: x.shift(1).rolling(10).std())
)
ticker_daily['volume_z'] = (
    (ticker_daily['volume'] - ticker_daily['vol_10d_mean'])
    / ticker_daily['vol_10d_std']
)

# 8. neg_extreme: bottom decile sentiment flag (global for EDA; recomputed per fold later)
_global_threshold = ticker_daily['mean_sent'].quantile(0.10)
ticker_daily['neg_extreme'] = (ticker_daily['mean_sent'] <= _global_threshold).astype(float)

# 9. neg_extreme_x_momentum: the interaction term
ticker_daily['neg_extreme_x_momentum'] = ticker_daily['neg_extreme'] * ticker_daily['mom_1d']

# Target: binary return direction
ticker_daily['target'] = (ticker_daily['ret'] > 0).astype(int)

print(f"Neg extreme threshold: {_global_threshold:.4f}")
print(f"Neg extreme days: {ticker_daily['neg_extreme'].sum():.0f} / {len(ticker_daily)}")
print(f"\nTarget distribution:\n{ticker_daily['target'].value_counts(normalize=True)}")

Neg extreme threshold: -0.1969
Neg extreme days: 139 / 1386

Target distribution:
target
1    0.510823
0    0.489177
Name: proportion, dtype: float64


In [7]:
feature_cols = [
    'mom_1d', 'mom_5d', 'vol_10d', 'mean_sent', 'sent_5d_avg', 'sent_surprise', 'volume_z', 'neg_extreme', 'neg_extreme_x_momentum'
]

target = 'target'

clean = ticker_daily.dropna(subset=feature_cols+[target]).copy()
print(f"Clean rows: {len(clean):,} (dropped {len(ticker_daily) - len(clean):,} due to NaN)")

Clean rows: 1,316 (dropped 70 due to NaN)


In [8]:
for col in feature_cols: 
    rho, p = spearmanr(clean[col], clean['ret'])
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "  "
    print(f"  {col:30s}  rho={rho:+.4f}  p={p:.4f}  {sig}")


  mom_1d                          rho=-0.0190  p=0.4910    
  mom_5d                          rho=-0.0411  p=0.1360    
  vol_10d                         rho=-0.0467  p=0.0901    
  mean_sent                       rho=-0.0511  p=0.0640    
  sent_5d_avg                     rho=-0.0063  p=0.8180    
  sent_surprise                   rho=-0.0347  p=0.2088    
  volume_z                        rho=+0.0570  p=0.0386  *
  neg_extreme                     rho=+0.0275  p=0.3183    
  neg_extreme_x_momentum          rho=-0.0144  p=0.6028    
